# Deepfake Detection Challenge_3

- backbone: `ResNet50` + 부분 fine-tuning
- `AdamW + CosineAnnealingLR + AMP`
- dropout 추가
- validation AUC 기준 best checkpoint 저장
- 테스트 시 `horizontal flip TTA` + 선택적 `face crop TTA`

## 학습 데이터셋

기존 baseline과 동일하게 **Kaggle의 `xhlulu/140k-real-and-fake-faces`** 데이터셋을 사용

- 총 140,000장
- real 70,000장 / fake 70,000장
- 라벨 정의
  - **real = 0**
  - **fake = 1**

In [1]:
# ============================================================
# 1. 필수 패키지 설치
# ============================================================

!pip cache purge
!pip install --user -q kagglehub pillow==11.1.0 --ignore-installed
!pip install --user -q facenet-pytorch --no-deps

Files removed: 72 (7.9 MB)
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.14.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.26a0+c5e1555-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dill-0.3.9-py

In [2]:
# ============================================================
# 2. 라이브러리 / 경로 / 하이퍼파라미터 설정
# ============================================================
import os
import io
import gc
import math
import json
import zipfile
import random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile, ImageOps
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, datasets

from sklearn.metrics import roc_auc_score

import kagglehub

ImageFile.LOAD_TRUNCATED_IMAGES = True


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False


def find_existing_path(candidates):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    raise FileNotFoundError(f"파일을 찾지 못했습니다. candidates={candidates}")


SEED = 42
seed_everything(SEED)

# -------------------------------
# 경로
# -------------------------------
WORKING_DIR = Path(os.getcwd())

COMP_ZIP_PATH = find_existing_path([
    "deepfake-detection-challenge-deep-preview-x-ai-7th.zip",
    "/mnt/data/deepfake-detection-challenge-deep-preview-x-ai-7th.zip",
])
EXTRACT_DIR = Path("competition_data")

DATASET_SLUG = "xhlulu/140k-real-and-fake-faces"
DATASET_DIR = None

# -------------------------------
# 학습 설정
# -------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 256
BATCH_SIZE = 32 if DEVICE == "cuda" else 8
NUM_WORKERS = min(4, os.cpu_count() or 2)

# baseline보다 더 넓게 사용
# 시간이 부족하면 TRAIN_PER_CLASS=3000, VAL_PER_CLASS=500 정도로 줄여도 됩니다.
TRAIN_PER_CLASS = 10000
VAL_PER_CLASS = 2000

EPOCHS = 5
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.02
DROPOUT = 0.30
USE_AMP = DEVICE == "cuda"

USE_FACE_CROP_TTA = True
FACE_MARGIN_RATIO = 0.25
MAX_FACE_DETECT_SIZE = 1600  # 너무 큰 원본은 detection용으로만 축소

# 제출 파일은 기본적으로 0~1 확률값을 저장합니다.
# ROC-AUC 기준에서는 연속 확률값 제출이 더 적절합니다.
WRITE_BINARY_SUBMISSION = False
BINARY_THRESHOLD = 0.5

print("WORKING_DIR:", WORKING_DIR)
print("COMP_ZIP_PATH:", COMP_ZIP_PATH)
print("DEVICE:", DEVICE)
print("IMAGE_SIZE:", IMAGE_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)

WORKING_DIR: /home/work
COMP_ZIP_PATH: deepfake-detection-challenge-deep-preview-x-ai-7th.zip
DEVICE: cuda
IMAGE_SIZE: 256
BATCH_SIZE: 32


In [3]:
# ============================================================
# 3. 대회 데이터 압축 해제
# ============================================================
assert Path(COMP_ZIP_PATH).exists(), f"압축 파일이 없습니다: {COMP_ZIP_PATH}"

if EXTRACT_DIR.exists():
    import shutil
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(COMP_ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("압축 해제 완료:", EXTRACT_DIR)
print("파일 목록:", os.listdir(EXTRACT_DIR))

압축 해제 완료: competition_data
파일 목록: ['sample_submission.csv', 'test.csv', 'test']


In [4]:
# ============================================================
# 4. 테스트 데이터와 제출 형식 불러오기
# ============================================================
TEST_DIR = EXTRACT_DIR / "test"
TEST_CSV_PATH = EXTRACT_DIR / "test.csv"
SAMPLE_SUB_PATH = EXTRACT_DIR / "sample_submission.csv"

assert TEST_DIR.exists(), "test 폴더가 없습니다."
assert TEST_CSV_PATH.exists(), "test.csv가 없습니다."
assert SAMPLE_SUB_PATH.exists(), "sample_submission.csv가 없습니다."

test_df = pd.read_csv(TEST_CSV_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print(test_df.head())
print(sample_sub.head())
print("테스트 이미지 수:", len(test_df))

        id         file_path
0  test001  test/test001.png
1  test002  test/test002.png
2  test003  test/test003.png
3  test004  test/test004.png
4  test005  test/test005.png
        id  score
0  test001    0.5
1  test002    0.5
2  test003    0.5
3  test004    0.5
4  test005    0.5
테스트 이미지 수: 300



## Kaggle 데이터셋 다운로드

이 셀은 기존 baseline과 동일하게 **GAN 기반 얼굴 데이터셋**을 내려받습니다.  
Colab에서 처음 실행하면 Kaggle 인증이 필요할 수 있습니다.


In [5]:
# ============================================================
# 5. 학습 데이터 다운로드
# ============================================================
DATASET_DIR = Path(kagglehub.dataset_download(DATASET_SLUG))
print("Downloaded dataset to:", DATASET_DIR)

for p in DATASET_DIR.iterdir():
    print(p.name)

Downloaded dataset to: /home/work/.cache/kagglehub/datasets/xhlulu/140k-real-and-fake-faces/versions/2
valid.csv
test.csv
real_vs_fake
train.csv


In [6]:
# ============================================================
# 6. 이미지 전처리 정의
# ============================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class RandomJPEGCompression:
    def __init__(self, p=0.20, quality_range=(70, 100)):
        self.p = p
        self.quality_range = quality_range

    def __call__(self, img):
        if random.random() > self.p:
            return img
        quality = random.randint(*self.quality_range)
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")


def pil_rgb_loader(path):
    with open(path, "rb") as f:
        img = Image.open(f)
        img = ImageOps.exif_transpose(img)
        return img.convert("RGB")


train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.05, hue=0.02),
    RandomJPEGCompression(p=0.20, quality_range=(70, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

valid_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [7]:
# ============================================================
# 7. 학습 데이터셋 구성
#    우선순위:
#    1) DATASET_DIR 내부의 real/fake 폴더에서 이미지 수집
#    2) 실패 시 DATASET_DIR 내부의 labeled csv 탐색
#
#    주의:
#    - competition_data/test.csv
#    - competition_data/sample_submission.csv
#    위 두 파일은 학습용 라벨 데이터가 아닙니다.
# ============================================================
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def is_image_file(path):
    return Path(path).suffix.lower() in IMG_EXTENSIONS


def collect_from_real_fake_dirs(root: Path):
    rows = []
    seen = set()

    label_dirs = []
    if root.is_dir() and root.name.lower() in {"real", "fake"}:
        label_dirs.append(root)
    label_dirs.extend([p for p in root.rglob("*") if p.is_dir() and p.name.lower() in {"real", "fake"}])

    for label_dir in label_dirs:
        label_name = label_dir.name.lower()
        label = 0 if label_name == "real" else 1

        for img_path in label_dir.rglob("*"):
            if img_path.is_file() and is_image_file(img_path):
                key = str(img_path.resolve())
                if key in seen:
                    continue
                seen.add(key)
                rows.append({
                    "image_path": key,
                    "label": label,
                    "source": "folder",
                })

    return pd.DataFrame(rows)


def normalize_binary_label(v):
    if pd.isna(v):
        return None

    if isinstance(v, str):
        s = v.strip().lower()
        if s in {"real", "0", "false", "no", "clean"}:
            return 0
        if s in {"fake", "1", "true", "yes", "deepfake"}:
            return 1

    try:
        x = float(v)
        if x == 0:
            return 0
        if x == 1:
            return 1
    except Exception:
        pass

    return None


def collect_from_labeled_csvs(root: Path):
    rows = []

    csv_paths = sorted(root.rglob("*.csv"))
    print("외부 데이터셋 내부 csv 후보 수:", len(csv_paths))
    if csv_paths:
        preview = [str(p.relative_to(root)) for p in csv_paths[:10]]
        print("csv 예시:", preview)

    path_col_candidates = [
        "image_path", "file_path", "filepath", "path", "filename", "image", "img_path", "img"
    ]
    label_col_candidates = [
        "label", "target", "class", "y", "is_fake", "fake", "real_or_fake"
    ]

    for csv_path in csv_paths:
        try:
            df = pd.read_csv(csv_path, low_memory=False)
        except Exception:
            continue

        lower_map = {str(c).lower(): c for c in df.columns}
        path_col = next((lower_map[c] for c in path_col_candidates if c in lower_map), None)
        label_col = next((lower_map[c] for c in label_col_candidates if c in lower_map), None)

        if path_col is None or label_col is None:
            continue

        kept = 0
        for row in df[[path_col, label_col]].itertuples(index=False):
            raw_path, raw_label = row
            label = normalize_binary_label(raw_label)
            if label is None or pd.isna(raw_path):
                continue

            rel = Path(str(raw_path).replace("\\", "/"))
            candidates = [
                csv_path.parent / rel,
                root / rel,
            ]

            resolved = None
            for cand in candidates:
                if cand.exists() and cand.is_file() and is_image_file(cand):
                    resolved = str(cand.resolve())
                    break

            if resolved is None:
                continue

            rows.append({
                "image_path": resolved,
                "label": label,
                "source": f"csv:{csv_path.name}",
            })
            kept += 1

        if kept > 0:
            print(f"라벨 csv 사용: {csv_path} | path_col={path_col} | label_col={label_col} | rows={kept}")

    if not rows:
        return pd.DataFrame(columns=["image_path", "label", "source"])

    meta = pd.DataFrame(rows).drop_duplicates(subset=["image_path"]).reset_index(drop=True)
    return meta


train_meta = collect_from_real_fake_dirs(DATASET_DIR)

if train_meta.empty:
    print("real/fake 폴더 구조를 찾지 못해, 외부 데이터셋 내부 csv 기반 labeled data를 탐색합니다.")
    train_meta = collect_from_labeled_csvs(DATASET_DIR)

assert not train_meta.empty, (
    "외부 학습 데이터셋에서 라벨된 이미지를 찾지 못했습니다. "
    f"DATASET_DIR={DATASET_DIR}"
)

train_meta = train_meta.drop_duplicates(subset=["image_path"]).reset_index(drop=True)
train_meta["label"] = train_meta["label"].astype(int)

print("전체 학습 메타 수:", len(train_meta))
print(train_meta["label"].value_counts().sort_index().rename(index={0: "real", 1: "fake"}))
print(train_meta.head())

전체 학습 메타 수: 140000
label
real    70000
fake    70000
Name: count, dtype: int64
                                          image_path  label  source
0  /home/work/.cache/kagglehub/datasets/xhlulu/14...      1  folder
1  /home/work/.cache/kagglehub/datasets/xhlulu/14...      1  folder
2  /home/work/.cache/kagglehub/datasets/xhlulu/14...      1  folder
3  /home/work/.cache/kagglehub/datasets/xhlulu/14...      1  folder
4  /home/work/.cache/kagglehub/datasets/xhlulu/14...      1  folder


In [8]:
# ============================================================
# 8. 학습/검증 분할
#    최종 학습 라벨은 real=0, fake=1 입니다.
# ============================================================
real_meta = train_meta[train_meta["label"] == 0].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
fake_meta = train_meta[train_meta["label"] == 1].sample(frac=1.0, random_state=SEED).reset_index(drop=True)

available_per_class = min(len(real_meta), len(fake_meta))
assert available_per_class >= 2, (
    f"학습에 필요한 이미지 수가 부족합니다. real={len(real_meta)}, fake={len(fake_meta)}"
)

actual_train_per_class = min(TRAIN_PER_CLASS, max(1, available_per_class - 1))
remaining_after_train = available_per_class - actual_train_per_class
actual_val_per_class = min(VAL_PER_CLASS, remaining_after_train)

if actual_val_per_class == 0:
    # 데이터가 너무 적을 때는 validation 1장 이상 확보
    actual_val_per_class = min(max(1, available_per_class // 10), available_per_class - 1)
    actual_train_per_class = available_per_class - actual_val_per_class

assert actual_train_per_class > 0 and actual_val_per_class > 0, (
    f"split 실패: available_per_class={available_per_class}, "
    f"train={actual_train_per_class}, val={actual_val_per_class}"
)

train_df = pd.concat([
    real_meta.iloc[:actual_train_per_class],
    fake_meta.iloc[:actual_train_per_class],
], axis=0).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

val_df = pd.concat([
    real_meta.iloc[actual_train_per_class:actual_train_per_class + actual_val_per_class],
    fake_meta.iloc[actual_train_per_class:actual_train_per_class + actual_val_per_class],
], axis=0).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("train real:", int((train_df['label'] == 0).sum()))
print("train fake:", int((train_df['label'] == 1).sum()))
print("val real:", int((val_df['label'] == 0).sum()))
print("val fake:", int((val_df['label'] == 1).sum()))

train real: 10000
train fake: 10000
val real: 2000
val fake: 2000


In [9]:
# ============================================================
# 9. Path 기반 Dataset
# ============================================================
class PathLabelDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = pil_rgb_loader(row["image_path"])
        if self.transform is not None:
            x = self.transform(x)
        y = torch.tensor(float(row["label"]), dtype=torch.float32)
        return x, y


train_dataset = PathLabelDataset(train_df, transform=train_tfms)
val_dataset = PathLabelDataset(val_df, transform=valid_tfms)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

Train size: 20000
Val size: 4000


In [10]:
# ============================================================
# 10. DataLoader
# ============================================================
loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_kwargs,
)

In [11]:
# ============================================================
# 11. 모델 정의
# - ResNet50 backbone
# - 앞단만 freeze
# - classifier에 dropout 추가
# ============================================================
weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
model = torchvision.models.resnet50(weights=weights)

# baseline처럼 완전 freeze는 하지 않고,
# 너무 일반적인 저수준 feature만 고정하고 나머지는 fine-tuning
for name, p in model.named_parameters():
    if name.startswith(("conv1", "bn1", "layer1")):
        p.requires_grad = False

in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(DROPOUT),
    nn.Linear(in_features, 1),
)
model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=LR * 0.1,
)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(model.fc)
print(f"Trainable params: {trainable_params:,} / Total params: {all_params:,}")

Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=2048, out_features=1, bias=True)
)
Trainable params: 23,284,737 / Total params: 23,510,081


/tmp/ipykernel_810/290571809.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [12]:
# ============================================================
# 12. 학습 / 검증 유틸리티
# ============================================================
BEST_CKPT_PATH = EXTRACT_DIR / "best_model_resnet50_improved.pth"
HISTORY_PATH = EXTRACT_DIR / "train_history_resnet50_improved.csv"


def smooth_targets(y, smoothing=0.0):
    if smoothing <= 0:
        return y
    return y * (1.0 - smoothing) + 0.5 * smoothing


@torch.no_grad()
def evaluate_auc(model, loader):
    model.eval()
    all_probs = []
    all_labels = []
    running_loss = 0.0

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(x).squeeze(1)
            loss = criterion(logits, y)

        probs = torch.sigmoid(logits)
        running_loss += loss.item() * x.size(0)

        all_probs.extend(probs.detach().cpu().numpy().tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    auc = roc_auc_score(all_labels, all_probs)
    avg_loss = running_loss / len(loader.dataset)
    return avg_loss, auc


def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0

    for x, y in tqdm(loader, desc="Train", leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        y_train = smooth_targets(y, LABEL_SMOOTHING)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(x).squeeze(1)
            loss = criterion(logits, y_train)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * x.size(0)

    return running_loss / len(loader.dataset)

In [13]:
# ============================================================
# 13. 학습
# ============================================================
best_auc = -np.inf
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_loss, val_auc = evaluate_auc(model, val_loader)
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc,
        "lr": current_lr,
    })

    print(f"[Epoch {epoch}/{EPOCHS}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_auc={val_auc:.5f} | lr={current_lr:.6f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            "epoch": epoch,
            "best_auc": best_auc,
            "model_state_dict": model.state_dict(),
        }, BEST_CKPT_PATH)
        print(f"  -> best checkpoint saved to {BEST_CKPT_PATH}")

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)
print("Saved history to:", HISTORY_PATH)
print("Best val AUC:", best_auc)

# best checkpoint 로드
ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print("Loaded best checkpoint from epoch:", ckpt["epoch"])


gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

Train:   0%|          | 0/625 [00:00<?, ?it/s]

/tmp/ipykernel_810/103299792.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_810/103299792.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


[Epoch 1/5] train_loss=0.2026 | val_loss=0.3734 | val_auc=0.99688 | lr=0.000274
  -> best checkpoint saved to competition_data/best_model_resnet50_improved.pth


Train:   0%|          | 0/625 [00:00<?, ?it/s]

/tmp/ipykernel_810/103299792.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_810/103299792.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


...

/tmp/ipykernel_810/103299792.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_810/103299792.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


[Epoch 4/5] train_loss=0.0687 | val_loss=0.0300 | val_auc=0.99999 | lr=0.000056
  -> best checkpoint saved to competition_data/best_model_resnet50_improved.pth


Train:   0%|          | 0/625 [00:00<?, ?it/s]

/tmp/ipykernel_810/103299792.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_810/103299792.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


[Epoch 5/5] train_loss=0.0638 | val_loss=0.0145 | val_auc=0.99999 | lr=0.000030
  -> best checkpoint saved to competition_data/best_model_resnet50_improved.pth
Saved history to: competition_data/train_history_resnet50_improved.csv
Best val AUC: 0.99999225
Loaded best checkpoint from epoch: 5


In [14]:
# ============================================================
# 14. 테스트셋 추론 준비
#     - 기본 full image view
#     - horizontal flip TTA
#     - optional: face crop TTA
# ============================================================
face_detector = None

if USE_FACE_CROP_TTA:
    try:
        from facenet_pytorch import MTCNN
        face_detector = MTCNN(
            image_size=160,
            margin=0,
            keep_all=True,
            post_process=False,
            device=DEVICE,
        )
        print("Face detector loaded successfully.")
    except Exception as e:
        print("Face detector load failed. full-image TTA만 사용합니다.")
        print("Reason:", repr(e))
        face_detector = None


def load_pil_image(path):
    with open(path, "rb") as f:
        img = Image.open(f)
        img = ImageOps.exif_transpose(img)
        return img.convert("RGB")


def crop_largest_face(pil_img, detector, margin_ratio=0.25, detect_max_side=1600):
    if detector is None:
        return None

    detect_img = pil_img
    scale = 1.0

    if max(pil_img.size) > detect_max_side:
        scale = detect_max_side / max(pil_img.size)
        new_size = (max(1, int(pil_img.size[0] * scale)), max(1, int(pil_img.size[1] * scale)))
        detect_img = pil_img.resize(new_size)

    boxes, probs = detector.detect(detect_img)

    if boxes is None or len(boxes) == 0:
        return None

    # 가장 큰 얼굴 + confidence 기준으로 선택
    best_idx = 0
    best_score = -1.0
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        area = max(0.0, x2 - x1) * max(0.0, y2 - y1)
        prob = 0.0 if probs is None or probs[i] is None else float(probs[i])
        score = area * max(prob, 1e-6)
        if score > best_score:
            best_score = score
            best_idx = i

    x1, y1, x2, y2 = boxes[best_idx]

    if scale != 1.0:
        x1, y1, x2, y2 = x1 / scale, y1 / scale, x2 / scale, y2 / scale

    bw = x2 - x1
    bh = y2 - y1
    x1 = max(0, int(x1 - bw * margin_ratio))
    y1 = max(0, int(y1 - bh * margin_ratio))
    x2 = min(pil_img.size[0], int(x2 + bw * margin_ratio))
    y2 = min(pil_img.size[1], int(y2 + bh * margin_ratio))

    if x2 <= x1 or y2 <= y1:
        return None

    return pil_img.crop((x1, y1, x2, y2))


@torch.no_grad()
def predict_single_image(model, pil_img, transform, detector=None):
    model.eval()

    views = [pil_img, ImageOps.mirror(pil_img)]

    face_img = crop_largest_face(
        pil_img,
        detector=detector,
        margin_ratio=FACE_MARGIN_RATIO,
        detect_max_side=MAX_FACE_DETECT_SIZE,
    )
    if face_img is not None:
        views.extend([face_img, ImageOps.mirror(face_img)])

    batch = torch.stack([transform(v) for v in views]).to(DEVICE)

    with torch.cuda.amp.autocast(enabled=USE_AMP):
        logits = model(batch).squeeze(1)
        probs = torch.sigmoid(logits)

    return float(probs.mean().item())

Face detector loaded successfully.


In [15]:
# ============================================================
# 15. 테스트셋 추론
# ============================================================
def resolve_test_image_path(row_dict):
    candidates = []

    if "file_path" in row_dict and pd.notna(row_dict["file_path"]):
        raw = str(row_dict["file_path"]).replace("\\", "/")
        candidates.append(EXTRACT_DIR / raw)
        candidates.append(TEST_DIR / Path(raw).name)

    if "id" in row_dict and pd.notna(row_dict["id"]):
        sample_id = str(row_dict["id"])
        for ext in [".png", ".jpg", ".jpeg", ".webp", ".bmp"]:
            candidates.append(TEST_DIR / f"{sample_id}{ext}")

    seen = set()
    for cand in candidates:
        cand = Path(cand)
        key = str(cand)
        if key in seen:
            continue
        seen.add(key)
        if cand.exists():
            return cand

    raise FileNotFoundError(f"테스트 이미지 경로를 찾지 못했습니다: {row_dict}")


pred_ids = []
pred_scores = []

for row in tqdm(test_df.to_dict(orient="records"), total=len(test_df), desc="Predicting test set"):
    img_path = resolve_test_image_path(row)
    pil_img = load_pil_image(img_path)
    score = predict_single_image(model, pil_img, valid_tfms, detector=face_detector)

    pred_ids.append(row["id"])
    pred_scores.append(score)

print("예측 완료:", len(pred_ids))
print("score min/max:", float(np.min(pred_scores)), float(np.max(pred_scores)))

Predicting test set:   0%|          | 0/300 [00:00<?, ?it/s]

/tmp/ipykernel_810/4198353955.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


예측 완료: 300
score min/max: 0.00446319580078125 0.484375


In [16]:
# ============================================================
# 16. 제출 파일 생성
# ============================================================
submission = sample_sub[["id"]].copy()

pred_map = dict(zip(pred_ids, pred_scores))
submission["score"] = submission["id"].map(pred_map)

assert submission["score"].notnull().all(), "누락된 예측값이 있습니다."

submission["score"] = submission["score"].astype(float).clip(0.0, 1.0)

submission_path = EXTRACT_DIR / "submission_3.csv"
submission.to_csv(submission_path, index=False)

print("Saved probability submission to:", submission_path)
print(submission.head())

if WRITE_BINARY_SUBMISSION:
    binary_submission = submission.copy()
    binary_submission["score"] = (binary_submission["score"] >= BINARY_THRESHOLD).astype(int)
    binary_path = EXTRACT_DIR / "submission_3_binary.csv"
    binary_submission.to_csv(binary_path, index=False)
    print("Saved binary submission to:", binary_path)
    print(binary_submission.head())

Saved probability submission to: competition_data/submission_3.csv
        id     score
0  test001  0.017838
1  test002  0.009811
2  test003  0.016174
3  test004  0.012131
4  test005  0.011490


In [17]:
# mv submission_3.csv /home/work/COMPETITION/

In [18]:
# mv Deep_Preview_3.ipynb /home/work/COMPETITION/